In [13]:
from qiskit_ibm_runtime.fake_provider import FakeFez, FakeTorino, FakeMarrakesh, FakePerth
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit import QuantumCircuit
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit_aer import AerSimulator

In [24]:
backend = FakeFez()

coupling_map = backend.coupling_map
two_qubit_gate = 'ecr' if 'ecr' in backend.operation_names else 'cz'

error_map = {}
for edge in coupling_map.get_edges():

        error_map[edge] = backend.target[two_qubit_gate][edge].error


print("Error map:", error_map)

Error map: {(0, 1): 0.0101566715843735, (1, 0): 0.0101566715843735, (1, 2): 0.0037717624378870163, (2, 1): 0.0037717624378870163, (2, 3): 0.0029846582394649213, (3, 2): 0.0029846582394649213, (3, 4): 0.004349994010613423, (3, 16): 0.004177743557862323, (4, 3): 0.004349994010613423, (4, 5): 0.00276468082328718, (5, 4): 0.00276468082328718, (5, 6): 0.014920610187955818, (6, 5): 0.014920610187955818, (6, 7): 0.004585711051097008, (7, 6): 0.004585711051097008, (7, 8): 0.004850251291634411, (7, 17): 0.004258349250875926, (8, 7): 0.004850251291634411, (8, 9): 0.0029547288488357415, (9, 8): 0.0029547288488357415, (9, 10): 0.0034837086548570873, (10, 9): 0.0034837086548570873, (10, 11): 0.004010049046041353, (11, 10): 0.004010049046041353, (11, 12): 0.003964835764014463, (11, 18): 0.003578834927444652, (12, 11): 0.003964835764014463, (12, 13): 0.003090315212871342, (13, 12): 0.003090315212871342, (13, 14): 0.0038150866361409963, (14, 13): 0.0038150866361409963, (14, 15): 0.0057363415360358605,

In [41]:
def bfs_sum_error_depth(start, coupling_map, depth, qubit_scores):
    visited = set()
    queue = [(start, 0)]  # (node, current_depth)
    area_score = 1

    while queue:
        node, current_depth = queue.pop(0)
        if node not in visited and current_depth <= depth:
            visited.add(node)
            area_score *= qubit_scores[node]
            
            neighbor_indices = [n for n in coupling_map.neighbors(node)]

            for neighbor in neighbor_indices:
                queue.append((neighbor, current_depth + 1))
    
    
    
    return area_score

In [ ]:
qubit_scores = {i: 1.0 for i in range(backend.num_qubits)}
# print("Initial qubit scores:", qubit_scores)

for edge, error in error_map.items():
    qubit_scores[edge[0]] *= (1 - error)
    qubit_scores[edge[1]] *= (1 - error)

best_area_queue = []

for i in range(backend.num_qubits):
    qubit_props = backend.properties().qubit_property(i)
    readout_error = qubit_props.get("readout_error", (None,))[0]
    qubit_scores[i] *= (1 - readout_error)
    qubit_scores[i] *= bfs_sum_error_depth(i, coupling_map, depth=2, qubit_scores=qubit_scores)


# print("Updated qubit scores:", qubit_scores)

Updated qubit scores: {0: 0.8999398666701064, 1: 0.8012682814535741, 2: 0.6590070239173045, 3: 0.44922478334053123, 4: 0.26200151495040597, 5: 0.10098861631728254, 6: 0.02225848513930986, 7: 0.0, 8: 0.0, 9: 0.0, 10: 0.0, 11: 0.0, 12: 0.0, 13: 0.0, 14: 0.0, 15: 0.0, 16: 0.07018144040957619, 17: 0.0, 18: 0.0, 19: 0.0, 20: 0.8931940336126099, 21: 0.7589390128973126, 22: 0.04327877842239504, 23: 0.0009630793131959633, 24: 2.652588236858855e-06, 25: 0.0, 26: 0.0, 27: 0.0, 28: 0.0, 29: 0.0, 30: 0.0, 31: 0.0, 32: 0.0, 33: 0.0, 34: 0.0, 35: 0.0, 36: 0.026577213392788172, 37: 0.0, 38: 0.0, 39: 0.0, 40: 0.02462898880843441, 41: 0.000411833383168526, 42: 2.402373466591949e-07, 43: 7.668442689782491e-11, 44: 0.0, 45: 0.0, 46: 0.0, 47: 0.0, 48: 0.0, 49: 0.0, 50: 0.0, 51: 0.0, 52: 0.0, 53: 0.0, 54: 0.0, 55: 0.0, 56: 0.0, 57: 0.0, 58: 0.0, 59: 0.0, 60: 0.9199394282352625, 61: 0.0, 62: 0.0, 63: 0.0, 64: 0.0, 65: 0.0, 66: 0.0, 67: 0.0, 68: 0.0, 69: 0.0, 70: 0.0, 71: 0.0, 72: 0.0, 73: 0.0, 74: 0.0, 75: 

In [30]:
print(qubit_scores)

highest_score_qubits = sorted(qubit_scores, key=qubit_scores.get, reverse=True)[:2]
print("Highest score qubits:", highest_score_qubits, "with scores:", [qubit_scores[q] for q in highest_score_qubits])

{0: 0.9685471094143898, 1: 0.9610172234976069, 2: 0.9817380067968016, 3: 0.9616866577439428, 4: 0.9844010415304209, 5: 0.9577195889630005, 6: 0.9549292648765451, 7: 0.9524943065948012, 8: 0.9803934222283087, 9: 0.9801956857015078, 10: 0.9819698449400118, 11: 0.9670945583046693, 12: 0.9626145651338165, 13: 0.9759064909917545, 14: 0.9774390796576571, 15: 0.9504154768192422, 16: 0.9715423584725305, 17: 0.9579734108864939, 18: 0.9796610695342083, 19: 0.9813554161914319, 20: 0.9786300154798012, 21: 0.9556619078920032, 22: 0.9784544587613792, 23: 0.9799468607661449, 24: 0.9791448001054848, 25: 0.9708350854478888, 26: 0.9656444696619011, 27: 0.0, 28: 0.0, 29: 0.9596712285825812, 30: 0.9656331717724973, 31: 0.0, 32: 0.0, 33: 0.0, 34: 0.9802354532536018, 35: 0.9804618847596025, 36: 0.9770593418333943, 37: 0.981815688696295, 38: 0.9800255688492189, 39: 0.9792833163351834, 40: 0.9854580908966823, 41: 0.9323039805442362, 42: 0.9691667130037217, 43: 0.9447136782549022, 44: 0.9846128187119477, 45: 0

In [39]:
neighbor_indices = [n for n in coupling_map.neighbors(0)]
print(neighbor_indices)

[1]
